# AWS SageMaker: ICU Readmission Predictive Modeling
## Comparative Analysis: Logistic Regression vs. LightGBM Classifier

This notebook trains and evaluates two machine learning models for 30-day ICU patient readmission prediction using cleaned telemetry data ingested from **AWS S3** (`s3://healthcare-analytics-pipeline-[your-name]/cleaned-data/`).

### Evaluation Metrics Tracked:
- Accuracy
- Precision
- Recall
- ROC-AUC Score

In [ ]:
# 1. Install & import necessary dependencies
!pip install --quiet pandas numpy scikit-learn lightgbm boto3 matplotlib seaborn

import io
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, roc_auc_score, 
    f1_score, confusion_matrix, roc_curve
)

print("Libraries successfully imported.")

### 2. Load Processed Dataset from AWS S3
Replace `[your-name]` with your unique S3 bucket identifier configured during Step 1.

In [ ]:
# S3 Configuration
bucket_name = "healthcare-analytics-pipeline-[your-name]"
s3_key = "cleaned-data/patient_telemetry_clean.csv"
s3_uri = f"s3://{bucket_name}/{s3_key}"

print(f"Fetching dataset from: {s3_uri}")
try:
    df = pd.read_csv(s3_uri)
except Exception as e:
    print(f"Falling back to local copy or simulation: {e}")
    df = pd.read_csv("../data/cleaned/patient_telemetry_clean.csv")

display(df.head())
print("Dataset shape:", df.shape)

### 3. Feature Selection & Train-Test Splitting
Features include normalized clinical telemetry vitals (`heart_rate_norm`, `blood_pressure_norm`, `glucose_norm`, `age_norm`) and comorbidity count (`underlying_conditions`).

In [ ]:
feature_cols = ['heart_rate_norm', 'blood_pressure_norm', 'glucose_norm', 'age_norm', 'underlying_conditions']
target_col = 'readmitted_30days'

X = df[feature_cols]
y = df[target_col]

# 80% Training, 20% Testing split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

### 4. Train Model 1: Baseline Logistic Regression

In [ ]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)

lr_preds = lr_model.predict(X_test)
lr_probs = lr_model.predict_proba(X_test)[:, 1]
print("Logistic Regression Training Completed.")

### 5. Train Model 2: LightGBM Gradient Boosted Decision Tree

In [ ]:
lgb_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    verbose=-1
)
lgb_model.fit(X_train, y_train)

lgb_preds = lgb_model.predict(X_test)
lgb_probs = lgb_model.predict_proba(X_test)[:, 1]
print("LightGBM Training Completed.")

### 6. Side-by-Side Performance Comparison Matrix

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'LightGBM Classifier'],
    'Accuracy': [accuracy_score(y_test, lr_preds), accuracy_score(y_test, lgb_preds)],
    'Precision': [precision_score(y_test, lr_preds), precision_score(y_test, lgb_preds)],
    'Recall': [recall_score(y_test, lr_preds), recall_score(y_test, lgb_preds)],
    'F1-Score': [f1_score(y_test, lr_preds), f1_score(y_test, lgb_preds)],
    'ROC-AUC': [roc_auc_score(y_test, lr_probs), roc_auc_score(y_test, lgb_probs)]
})

display(results.round(4))

# ROC Curve Plot
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
fpr_lgb, tpr_lgb, _ = roc_curve(y_test, lgb_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, lr_probs):.3f})')
plt.plot(fpr_lgb, tpr_lgb, label=f'LightGBM Classifier (AUC = {roc_auc_score(y_test, lgb_probs):.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Chance Line')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve Comparison')
plt.legend()
plt.grid(True)
plt.show()